# Main Code to carry out blocking statistics and diagnostics on model data (CESM+others) and observations

### Specific Code to:
#### 1. Setup the analysis to be done and data to be ingested
#### 2. Climatology plots
#### 3. Jet latitude index
#### 4. 1D Frequency plots
#### 5. 2d Frequency plots
#### 6. PDF of blocking strengths
#### 7. Composite of plots conditioned on blocking strength
#### 8. PDF of blocking/baroclinic events length

In [1]:
# IMPORT PACKAGES

import importlib
import blocking_utils as block_utils
import blocking_figs as block_figs

%load_ext autoreload
%autoreload 2

In [2]:
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [3]:
cluster = PBSCluster(
    account="P03010039",
    interface="ext",
    walltime="12:00:00",
    queue="main",   
    cores=4,
    memory="32GB",
    processes=4,      # one process per core (safe default)cesm2
    local_directory="/glade/derecho/scratch/rneale/dask-temp",
    log_directory="/glade/derecho/scratch/rneale/dask-logs"
)

cluster.scale(jobs=32)
client = Client(cluster)

client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Compute/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Compute/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.209:39083,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Compute/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


2026-01-31 08:44:44,610 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='jupyterhub.hpc.ucar.edu', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/glade/u/apps/opt/conda/envs/npl-2025b/lib/python3.12/site-packages/tornado/web.py", line 1848, in _execute
    result = await result
             ^^^^^^^^^^^^
  File "/glade/u/apps/opt/conda/envs/npl-2025b/lib/python3.12/site-packages/tornado/websocket.py", line 277, in get
    await self.ws_connection.accept_connection(self)
  File "/glade/u/apps/opt/conda/envs/npl-2025b/lib/python3.12/site-packages/tornado/websocket.py", line 890, in accept_connection
    await self._accept_connection(handler)
  File "/glade/u/apps/opt/conda/envs/npl-2025b/lib/python3.12/site-packages/tornado/websocket.py", line 973, in _accept_connection
    await self._receive_frame_loop()
  File "/glade/u/apps/opt/conda/envs/npl-20


# TOP LEVEL OPTIONS


In [61]:
ens_names = ['ERA5','MERRA','CESM1','CESM2','b.e30_alpha07g.BHISTC_LTso.ne30_t232_wgx3.271','b.e30_alpha07g.BHISTC_LTso.ne30_t232_wgx3.276']
ens_mem_num = [1,1,20,20,1,1]  # Ensemble members per ensemble set
#ens_ystart = ['1979']*len(ens_names)
#ens_yend     = ['2005']*len(ens_names)

ens_ystart = ['1979','1979','1979','1979','1979','1979'] # Less the 4 digit years have to be prepended by zeros here (mostly 1850 coupled runs then)
ens_yend     = ['2005','2005','2005','2005','2005','2005']

block_season = 'JJA'
cam_var = 'Z500'
block_diag_hem = ['nhem']  
block_diag_set = ['1d']


# MAIN ROUTINES


In [62]:
importlib.reload(block_utils) # Required because I am constantly editing utility routines.

# Grab basic ens+run information and populate a dictionary)
block_meta = block_utils.ens_setup(ens_names,ens_mem_num,ens_ystart,ens_yend)

 
++ Large ensemble =  CESM1 :  20  out of  42  total (first/last) ++
b.e11.B20TRC5CNBDRD.f09_g16.001
b.e11.B20TRC5CNBDRD.f09_g16.020
 
++ Large ensemble =  CESM2 :  20  out of  50  total (first/last) ++
b.e21.BHISTcmip6.f09_g17.LE2-1001.001
b.e21.BHISTcmip6.f09_g17.LE2-1231.010


# Grab daily datasets needed for each 'ensemble'

In [63]:
ensemble_ds = block_utils.dataset_get(block_meta,cam_var,block_season,block_diag_hem)

-> dataset_get -> Requested season     :  JJA
-> dataset_get ->  Opening ensemble  ERA5  -  1  ensemble(s)
-> dataset_get -> Requested year range :  1979 - 2005
-> dataset_get ->  Opening ensemble  MERRA  -  1  ensemble(s)
-> dataset_get -> Requested year range :  1979 - 2005
-> dataset_get ->  Opening ensemble  CESM1  -  20  ensemble(s)
-> dataset_get -> Requested year range :  1979 - 2005
-> dataset_get ->  Opening ensemble  CESM2  -  20  ensemble(s)
-> dataset_get -> Requested year range :  1979 - 2005
-> dataset_get ->  Opening ensemble  CESM3-271  -  1  ensemble(s)
-> dataset_get -> Requested year range :  1979 - 2005
-> dataset_get ->  Opening ensemble  CESM3-276  -  1  ensemble(s)
-> dataset_get -> Requested year range :  1979 - 2005
-> dataset_get ->  Duration: 66.72582817077637



#  Calculate/Read/Write blocking frequency: 1D

In [64]:
ensemble_block_1d = block_utils.block_z500_freq(block_meta,ensemble_ds,block_season,block_diag = '1D', file_opts = 'w')

['ERA5', 'MERRA', 'CESM1', 'CESM2', 'CESM3-271', 'CESM3-276']
-> block_z500_freq ->   Calculating blocking statistics for  ERA5
-> block_file_read_write ->  Writing file for ensemble  ERA5  =  block_1D_ERA5_nens.1_1979-2005_JJA.nc
-> block_file_read_write ->  Done ...
-> block_z500_freq ->  Min/max blocking frequency for ensemble  ERA5  =  1.288244766505636 , 19.726247987117553
-> block_z500_freq ->   Calculating blocking statistics for  MERRA
-> block_file_read_write ->  Writing file for ensemble  MERRA  =  block_1D_MERRA_nens.1_1979-2005_JJA.nc
-> block_file_read_write ->  Done ...
-> block_z500_freq ->  Min/max blocking frequency for ensemble  MERRA  =  0.7648953301127215 , 17.19001610305958
-> block_z500_freq ->   Calculating blocking statistics for  CESM1
-> block_file_read_write ->  Writing file for ensemble  CESM1  =  block_1D_CESM1_nens.20_1979-2005_JJA.nc
-> block_file_read_write ->  Done ...
-> block_z500_freq ->  Min/max blocking frequency for ensemble  CESM1  =  0.362318840


# Calculate/Read/Write blocking frequency: 2D


In [ ]:
ensemble_block_2d = block_utils.block_z500_freq(block_meta,ensemble_ds,block_season,block_diag = '2D', file_opts = 'w')

['ERA5', 'MERRA', 'CESM1', 'CESM2', 'CESM3-271', 'CESM3-276']
-> block_z500_freq ->   Calculating blocking statistics for  ERA5
-> block_file_read_write ->  Writing file for ensemble  ERA5  =  block_2D_ERA5_nens.1_1979-2005_JJA.nc
-> block_file_read_write ->  Done ...
-> block_z500_freq ->  Min/max blocking frequency for ensemble  ERA5  =  0.0 , 95.93397745571659
-> block_z500_freq ->   Calculating blocking statistics for  MERRA
-> block_file_read_write ->  Writing file for ensemble  MERRA  =  block_2D_MERRA_nens.1_1979-2005_JJA.nc
-> block_file_read_write ->  Done ...
-> block_z500_freq ->  Min/max blocking frequency for ensemble  MERRA  =  0.0 , 83.33333333333334
-> block_z500_freq ->   Calculating blocking statistics for  CESM1
-> block_file_read_write ->  Writing file for ensemble  CESM1  =  block_2D_CESM1_nens.20_1979-2005_JJA.nc


/glade/u/apps/opt/conda/envs/npl-2025b/lib/python3.12/site-packages/distributed/client.py:3371: UserWarning: Sending large graph of size 11.33 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


-> block_file_read_write ->  Done ...


/glade/u/apps/opt/conda/envs/npl-2025b/lib/python3.12/site-packages/distributed/client.py:3371: UserWarning: Sending large graph of size 11.49 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(



# Plot 1D blocking: Shade = 1,2 +/- std (or min to max range)


In [ ]:
block_figs.block_plot_1d(block_meta,ensemble_block_1d,block_season,fig_out=True)


# Plot 2D blocking: Ensemble ave, 1 ensemble member, or all ensemble members 


In [ ]:
block_figs.block_plot_2d(block_meta,ensemble_block_2d,block_season,fig_out=True,ens_plot='av')


# Plot 1d regional blocking PDFs (European, Pacific and Greenland)


In [11]:
block_figs.block_plot_1d(block_meta,ensemble_block_1d,block_season,ens_plot='1',fig_out=True)

TypeError: block_plot_1d() got an unexpected keyword argument 'ens_plot'

In [ ]:
#bstats = atmos_block.stats
#bclimo = atmos_block.climo
#bjet_lat = atmos_block.get_lat
#bfreq_1d = atmos_block.bfreq_1d
#bfreq_2d = atmos_block.bfreq_2d
#bpdf_1d = atmos_block.bfreq_2d
#bcomp_lag = atmos_block.bfreq_lag
#bday_len = atmos_block.bday_len